# RISK ANALYTICS — 04 Capa Gold

**Origen:** `eventosriesgoyvar.silver.eventos_riesgo` (51.889 filas × 78 columnas, validada)
**Destino:** esquema `eventosriesgoyvar.gold` (tablas Delta)

Reglas de agregación (definidas sobre los datos reales de Silver):
- **Periodicidad:** mensual. `periodo` = primer día del mes (date), con `anio_evento` y `mes_evento` como claves. Las tablas por período usan un **calendario mensual completo** (dic-2014 a sep-2026, 142 meses): los meses sin eventos registrados quedan en 0 (convención para series completas, no imputación de datos). Años incompletos por diseño de la fuente: 2014 (solo diciembre, 2 eventos) y 2026 (hasta septiembre).
- **Frecuencia:** se cuentan TODOS los eventos (51.889), con o sin cuantía. Se desagrega también por código de pérdida A (insumo del VaR) y por eventos con recuperación.
- **Severidad:** se calcula SIEMPRE sobre la `cuantia_bruta` y SOLO en eventos con cuantía (26.701; 51,5%). El evento sin cuantía NO es severidad 0: figura como `SIN_CUANTIA` en las tablas de frecuencia y como NULL en indicadores y series.
- **Pérdidas:** sumas de cuantía bruta, neta y total recuperada por período y por segmento. **Segmento = categoría N1** del riesgo operacional (nivel BASA de mayor agregación disponible con nombre descriptivo).
- **Regla VaR vigente (20-Nov-2026):** los insumos de modelos de severidad/VaR usan SOLO `codigo_perdida = 'A'` (tipo R) con cuantía bruta: 2.046 eventos; los 21 eventos A sin cuantía quedan excluidos (no se imputan). Los eventos B (49.822) alimentan frecuencia y descriptivos.
- **Reconstrucción:** cada tabla Gold se recalcula completa (overwrite) desde Silver; nunca desde Bronze.
- En esta etapa NO se construyen modelos ni dashboard: solo tablas consolidadas, indicadores e insumos.

In [0]:
from pyspark.sql import functions as F
from datetime import datetime

TABLA_SILVER = "eventosriesgoyvar.silver.eventos_riesgo"
SCHEMA_GOLD = "eventosriesgoyvar.gold"
LOTE_GOLD = f"gold_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

spark.sql("CREATE SCHEMA IF NOT EXISTS eventosriesgoyvar.gold "
          "COMMENT 'Capa Gold: agregados consolidados e insumos para los modelos de riesgo operacional'")

silver = spark.table(TABLA_SILVER)
FILAS_SILVER = silver.count()

# Reglas de agregación reutilizables (definidas en el encabezado del notebook)
CON_CUANTIA = F.col("cuantia_bruta").isNotNull()   # severidad solo sobre cuantía bruta
ES_CODIGO_A = F.col("codigo_perdida") == "A"         # regla VaR: solo código pérdida A (tipo R)

def escribir(df, tabla, comentario):
    """Escribe una tabla Gold: reconstrucción completa (overwrite) + comentario + lote.
    overwriteSchema=True: al recalcular con percentiles exactos el tipo de una columna puede
    cambiar (p. ej. mediana pasa de decimal a double); el esquema se reemplaza por completo."""
    nombre = f"{SCHEMA_GOLD}.{tabla}"
    (df.withColumn("_lote_gold", F.lit(LOTE_GOLD))
       .write.option("overwriteSchema", "true").mode("overwrite").saveAsTable(nombre))
    spark.sql(f"COMMENT ON TABLE {nombre} IS '{comentario}'")
    print(f"{tabla:32s} {spark.table(nombre).count():>8,} filas")

print("Fuente:", TABLA_SILVER, "|", FILAS_SILVER, "filas | Lote:", LOTE_GOLD)

Fuente: eventosriesgoyvar.silver.eventos_riesgo | 51889 filas | Lote: gold_20261121_210730


In [0]:
# --- 1) Frecuencia por período: calendario mensual completo dic-2014 a sep-2026 ---
ini, fin = (silver.agg(F.min(F.date_trunc("MM", "fecha_inicio")).cast("date").alias("ini"),
                       F.max(F.date_trunc("MM", "fecha_inicio")).cast("date").alias("fin"))
            .collect()[0])
calendario = spark.sql(f"SELECT explode(sequence(to_date('{ini}'), to_date('{fin}'), interval 1 month)) AS periodo")
print(f"Calendario mensual: {ini} a {fin} ({calendario.count()} meses)")

frec_mes = (silver.groupBy("anio_evento", "mes_evento")
    .agg(F.count("*").alias("eventos"),
         F.sum(F.when(CON_CUANTIA, 1).otherwise(0)).alias("eventos_con_cuantia"),
         F.sum(F.when(ES_CODIGO_A, 1).otherwise(0)).alias("eventos_codigo_a"),
         F.sum(F.when(F.col("flg_con_recuperacion"), 1).otherwise(0)).alias("eventos_con_recuperacion"))
    .withColumn("periodo", F.to_date(F.concat(F.col("anio_evento").cast("string"),
             F.lpad(F.col("mes_evento").cast("string"), 2, "0"), F.lit("01")), "yyyyMMdd")))

freq_eventos_periodo = (calendario.alias("c")
    .join(frec_mes.alias("f"), F.col("c.periodo") == F.col("f.periodo"), "left")
    .select(F.col("c.periodo").alias("periodo"),
            F.year("c.periodo").alias("anio_evento"), F.month("c.periodo").alias("mes_evento"),
            F.coalesce(F.col("eventos"), F.lit(0)).alias("eventos"),
            F.coalesce(F.col("eventos_con_cuantia"), F.lit(0)).alias("eventos_con_cuantia"),
            F.coalesce(F.col("eventos_codigo_a"), F.lit(0)).alias("eventos_codigo_a"),
            F.coalesce(F.col("eventos_con_recuperacion"), F.lit(0)).alias("eventos_con_recuperacion"))
    .orderBy("periodo"))
escribir(freq_eventos_periodo, "freq_eventos_periodo",
         "Frecuencia mensual de eventos sobre calendario completo (dic-2014 a sep-2026); meses sin registros = 0 eventos")

# --- 2) Frecuencia por proceso, categoría, causa y severidad ---
def freq_por(cols, tabla, comentario):
    df = (silver.groupBy(*cols)
          .agg(F.count("*").alias("eventos"),
               F.sum(F.when(CON_CUANTIA, 1).otherwise(0)).alias("eventos_con_cuantia"),
               F.round(F.sum("cuantia_bruta"), 2).alias("suma_cuantia_bruta"))
          .withColumn("pct_eventos", F.round(F.col("eventos") / FILAS_SILVER * 100, 2)))
    escribir(df.orderBy(F.desc("eventos")), tabla, comentario)

freq_por(["codigo_proceso", "proceso"], "freq_por_proceso",
         "Frecuencia histórica por proceso (código y nombre homologados en Silver)")
freq_por(["cod_categoria_n1", "categoria_n1"], "freq_por_categoria",
         "Frecuencia histórica por categoría N1 del riesgo operacional")
freq_por(["cod_causa"], "freq_por_causa",
         "Frecuencia histórica por causa (solo código: la fuente no trae nombre descriptivo)")
freq_por([F.coalesce(F.col("nivel_severidad"), F.lit("SIN_CUANTIA")).alias("nivel_severidad")],
         "freq_por_severidad",
         "Frecuencia por nivel de severidad derivado; SIN_CUANTIA = evento sin cuantía bruta (no es severidad 0)")

Calendario mensual: 2014-12-01 a 2026-09-01 (142 meses)
freq_eventos_periodo                  142 filas
freq_por_proceso                       45 filas
freq_por_categoria                      7 filas
freq_por_causa                        524 filas
freq_por_severidad                      5 filas


In [0]:
# --- 3) Indicadores de severidad (solo eventos con cuantía bruta: 26.701) ---
# Percentiles EXACTOS (F.percentile): evita la desviación de percentile_approx
# (mediana aprox = 366.715 vs exacta = 367.371).
ESTADISTICOS = [
    F.count("*").alias("eventos_con_cuantia"),
    F.round(F.sum("cuantia_bruta"), 2).alias("suma_cuantia_bruta"),
    F.round(F.avg("cuantia_bruta"), 2).alias("promedio"),
    F.round(F.percentile("cuantia_bruta", 0.50), 2).alias("mediana"),
    F.round(F.percentile("cuantia_bruta", 0.75), 2).alias("p75"),
    F.round(F.percentile("cuantia_bruta", 0.90), 2).alias("p90"),
    F.round(F.percentile("cuantia_bruta", 0.95), 2).alias("p95"),
    F.round(F.percentile("cuantia_bruta", 0.99), 2).alias("p99"),
    F.round(F.stddev("cuantia_bruta"), 2).alias("desviacion_estandar"),
    F.min("cuantia_bruta").alias("minimo"),
    F.max("cuantia_bruta").alias("maximo"),
]
con_cuantia = silver.filter(CON_CUANTIA)

escribir(con_cuantia.agg(*ESTADISTICOS), "ind_severidad_global",
         "Indicadores de severidad del histórico completo: promedio, mediana, percentiles y extremos sobre cuantía bruta")
escribir(con_cuantia.groupBy("anio_evento").agg(*ESTADISTICOS).orderBy("anio_evento"), "ind_severidad_anual",
         "Indicadores de severidad por año (2014 y 2026 incompletos por diseño de la fuente)")

ind_severidad_global                    1 filas
ind_severidad_anual                    13 filas


In [0]:
# --- 4) Pérdidas históricas agregadas por período (mensual) y por segmento (categoría N1) ---
MEDIDAS_PERDIDA = [
    F.count("*").alias("eventos"),
    F.sum(F.when(CON_CUANTIA, 1).otherwise(0)).alias("eventos_con_cuantia"),
    F.coalesce(F.round(F.sum("cuantia_bruta"), 2), F.lit(0).cast("double")).alias("suma_cuantia_bruta"),
    F.coalesce(F.round(F.sum("cuantia_neta"), 2), F.lit(0).cast("double")).alias("suma_cuantia_neta"),
    F.coalesce(F.round(F.sum("cuantia_total_recuperada"), 2), F.lit(0).cast("double")).alias("suma_total_recuperada"),
]

perd_mes = (silver.groupBy("anio_evento", "mes_evento").agg(*MEDIDAS_PERDIDA)
    .withColumn("periodo", F.to_date(F.concat(F.col("anio_evento").cast("string"),
             F.lpad(F.col("mes_evento").cast("string"), 2, "0"), F.lit("01")), "yyyyMMdd")))

perdidas_por_periodo = (calendario.alias("c")
    .join(perd_mes.alias("p"), F.col("c.periodo") == F.col("p.periodo"), "left")
    .select(F.col("c.periodo").alias("periodo"),
            F.year("c.periodo").alias("anio_evento"), F.month("c.periodo").alias("mes_evento"),
            F.coalesce(F.col("eventos"), F.lit(0)).alias("eventos"),
            F.coalesce(F.col("eventos_con_cuantia"), F.lit(0)).alias("eventos_con_cuantia"),
            F.coalesce(F.col("suma_cuantia_bruta"), F.lit(0)).cast("double").alias("suma_cuantia_bruta"),
            F.coalesce(F.col("suma_cuantia_neta"), F.lit(0)).cast("double").alias("suma_cuantia_neta"),
            F.coalesce(F.col("suma_total_recuperada"), F.lit(0)).cast("double").alias("suma_total_recuperada"))
    .orderBy("periodo"))
escribir(perdidas_por_periodo, "perdidas_por_periodo",
         "Pérdidas mensuales agregadas: cuantía bruta, neta y total recuperada (COP); meses sin cuantías registradas = 0")

perdidas_por_segmento = (silver.groupBy("anio_evento", "cod_categoria_n1", "categoria_n1")
    .agg(*MEDIDAS_PERDIDA).orderBy("anio_evento", "cod_categoria_n1"))
escribir(perdidas_por_segmento, "perdidas_por_segmento",
         "Pérdidas anuales por segmento (categoría N1): cuantía bruta, neta y total recuperada (COP)")

perdidas_por_periodo                  142 filas
perdidas_por_segmento                  85 filas


In [0]:
# --- 5) Serie histórica mensual: frecuencia + severidad sobre calendario completo ---
serie_mes = (silver.groupBy("anio_evento", "mes_evento")
    .agg(F.count("*").alias("eventos"),
         F.sum(F.when(CON_CUANTIA, 1).otherwise(0)).alias("eventos_con_cuantia"),
         F.coalesce(F.round(F.sum("cuantia_bruta"), 2), F.lit(0)).cast("double").alias("suma_cuantia_bruta"),
         F.round(F.avg("cuantia_bruta"), 2).cast("double").alias("severidad_promedio"),
         F.round(F.percentile("cuantia_bruta", 0.50), 2).cast("double").alias("severidad_mediana"),
         F.round(F.percentile("cuantia_bruta", 0.95), 2).cast("double").alias("severidad_p95"),
         F.round(F.percentile("cuantia_bruta", 0.99), 2).cast("double").alias("severidad_p99"),
         F.max("cuantia_bruta").cast("double").alias("severidad_maximo"))
    .withColumn("periodo", F.to_date(F.concat(F.col("anio_evento").cast("string"),
             F.lpad(F.col("mes_evento").cast("string"), 2, "0"), F.lit("01")), "yyyyMMdd")))

series_frecuencia_severidad = (calendario.alias("c")
    .join(serie_mes.alias("s"), F.col("c.periodo") == F.col("s.periodo"), "left")
    .select(F.col("c.periodo").alias("periodo"),
            F.year("c.periodo").alias("anio_evento"), F.month("c.periodo").alias("mes_evento"),
            F.coalesce(F.col("eventos"), F.lit(0)).alias("eventos"),
            F.coalesce(F.col("eventos_con_cuantia"), F.lit(0)).alias("eventos_con_cuantia"),
            F.coalesce(F.col("suma_cuantia_bruta"), F.lit(0)).cast("double").alias("suma_cuantia_bruta"),
            F.col("severidad_promedio"), F.col("severidad_mediana"),
            F.col("severidad_p95"), F.col("severidad_p99"), F.col("severidad_maximo"))
    .orderBy("periodo"))
escribir(series_frecuencia_severidad, "series_frecuencia_severidad",
         "Serie histórica mensual de frecuencia y severidad (cuantía bruta); severidad NULL en meses sin eventos cuantificados (no se imputa)")

series_frecuencia_severidad           142 filas


In [0]:
# --- 6) Insumos para los modelos estadísticos ---
# Severidad (VaR): SOLO codigo_perdida='A' con cuantía bruta (regla del 20-Nov-2026).
# Los 21 eventos A sin cuantía quedan fuera del insumo de severidad (no se imputan).
modelo_insumo_severidad = (silver.filter(ES_CODIGO_A & CON_CUANTIA)
    .select("id_evento", "fecha_inicio", "anio_evento", "mes_evento", "codigo_perdida",
            "cuantia_bruta", "nivel_severidad", "cod_categoria_n1", "categoria_n1",
            "codigo_proceso", "proceso", "canal", "ciudad", "flg_requiere_revision")
    .orderBy("id_evento"))
escribir(modelo_insumo_severidad, "modelo_insumo_severidad",
         "Insumo del modelo de severidad/VaR: solo codigo_perdida A con cuantía bruta (2.046 eventos, regla acordada)")

# Frecuencia: panel mensual x categoría N1 (calendario completo; meses sin eventos = 0)
categorias = silver.select("cod_categoria_n1", "categoria_n1").distinct()
conteo_cat = (silver.groupBy("cod_categoria_n1", "anio_evento", "mes_evento")
    .agg(F.count("*").alias("eventos"))
    .withColumn("periodo", F.to_date(F.concat(F.col("anio_evento").cast("string"),
             F.lpad(F.col("mes_evento").cast("string"), 2, "0"), F.lit("01")), "yyyyMMdd")))

panel = categorias.crossJoin(calendario)  # 7 categorías x 142 meses
modelo_insumo_frecuencia = (panel.alias("p")
    .join(conteo_cat.alias("c"),
          (F.col("p.cod_categoria_n1") == F.col("c.cod_categoria_n1")) & (F.col("p.periodo") == F.col("c.periodo")),
          "left")
    .select(F.col("p.cod_categoria_n1"), F.col("p.categoria_n1"),
            F.col("p.periodo").alias("periodo"),
            F.year("p.periodo").alias("anio_evento"), F.month("p.periodo").alias("mes_evento"),
            F.coalesce(F.col("eventos"), F.lit(0)).alias("eventos"))
    .orderBy("cod_categoria_n1", "periodo"))
escribir(modelo_insumo_frecuencia, "modelo_insumo_frecuencia",
         "Panel mensual de frecuencia por categoría N1 (calendario completo): insumo para distribuciones de frecuencia")

modelo_insumo_severidad             2,046 filas
modelo_insumo_frecuencia              994 filas


In [0]:
# --- 7) Estructura para almacenar los resultados del VaR (se llena en la etapa de modelos) ---
spark.sql(f"""
CREATE OR REPLACE TABLE {SCHEMA_GOLD}.var_resultados (
  id_resultado BIGINT GENERATED ALWAYS AS IDENTITY COMMENT 'Identificador del resultado',
  fecha_calculo DATE COMMENT 'Fecha de ejecución del cálculo',
  metodo STRING COMMENT 'Método (p. ej. Monte Carlo LDA, empírico)',
  distribucion_frecuencia STRING COMMENT 'Distribución de frecuencia usada (p. ej. Poisson, Binomial Negativa)',
  distribucion_severidad STRING COMMENT 'Distribución de severidad usada (p. ej. Lognormal, empírica)',
  nivel_confianza DECIMAL(5,4) COMMENT 'Nivel de confianza (0.95, 0.99, 0.999)',
  horizonte STRING COMMENT 'Horizonte del VaR (mensual, anual)',
  segmento STRING COMMENT 'Segmento del cálculo (TOTAL o categoría/proceso)',
  valor_var DECIMAL(18,2) COMMENT 'Valor en Riesgo en COP',
  perdida_esperada DECIMAL(18,2) COMMENT 'Pérdida esperada en COP',
  n_simulaciones BIGINT COMMENT 'Número de simulaciones (si aplica)',
  n_eventos_insumo INT COMMENT 'Eventos usados como insumo del cálculo',
  notas STRING COMMENT 'Notas del cálculo',
  _lote STRING COMMENT 'Lote de procesamiento'
) USING DELTA
""")
spark.sql(f"COMMENT ON TABLE {SCHEMA_GOLD}.var_resultados IS "
          "'Estructura para almacenar los resultados del VaR: sin registros por ahora, se llena en la etapa de modelos'")
print(f"var_resultados creada (estructura vacía): {SCHEMA_GOLD}.var_resultados")

var_resultados creada (estructura vacía): eventosriesgoyvar.gold.var_resultados


In [0]:
# --- Validaciones: cada tabla Gold contra Silver y contra los conteos verificados ---
todo_ok = True
def check(nombre, ok, detalle=""):
    global todo_ok
    todo_ok = todo_ok and ok
    print(("OK   " if ok else "FALLA ") + nombre + (" | " + detalle if detalle else ""))

g = lambda t: spark.table(f"{SCHEMA_GOLD}.{t}")

# 1) Frecuencia: la suma de eventos de cada tabla == total de Silver
for t in ["freq_eventos_periodo", "freq_por_proceso", "freq_por_categoria", "freq_por_causa", "freq_por_severidad"]:
    total = g(t).agg(F.sum("eventos")).collect()[0][0]
    check(f"1. {t}: suma eventos == 51.889", total == FILAS_SILVER, f"suma={total}")
n_periodos = g("freq_eventos_periodo").count()
check("1b. freq_eventos_periodo: 142 meses (dic-2014 a sep-2026)", n_periodos == 142, f"meses={n_periodos}")
check("1c. freq_por_severidad: 5 niveles (BAJA/MEDIA/ALTA/EXTREMA + SIN_CUANTIA)", g("freq_por_severidad").count() == 5)
a_total = g("freq_eventos_periodo").agg(F.sum("eventos_codigo_a")).collect()[0][0]
check("1d. eventos código A en la serie == 2.067", a_total == 2067, f"total={a_total}")

# 2) Indicadores de severidad: contra los valores verificados en Silver
gi = g("ind_severidad_global").collect()[0]
check("2a. ind_severidad_global: 26.701 con cuantía", gi["eventos_con_cuantia"] == 26701, str(gi["eventos_con_cuantia"]))
mediana_exacta = silver.filter(CON_CUANTIA).agg(F.percentile("cuantia_bruta", 0.50).alias("m")).collect()[0]["m"]
check("2b. ind_severidad_global: mediana == percentil exacto de Silver (367.371)",
      float(gi["mediana"]) == float(mediana_exacta), f"gold={gi['mediana']}, exacta={mediana_exacta}")
check("2c. ind_severidad_global: suma == 241.789.756.996", float(gi["suma_cuantia_bruta"]) == 241789756996.00, str(gi["suma_cuantia_bruta"]))
anual = g("ind_severidad_anual").agg(F.sum("eventos_con_cuantia")).collect()[0][0]
check("2d. ind_severidad_anual: suma de eventos == 26.701", anual == 26701, f"suma={anual}")
check("2e. ind_severidad_anual: 13 años (2014-2026)", g("ind_severidad_anual").count() == 13)

# 3) Pérdidas: totales contra Silver
for t in ["perdidas_por_periodo", "perdidas_por_segmento"]:
    r = g(t).agg(F.sum("eventos").alias("e"), F.round(F.sum("suma_cuantia_bruta"), 2).alias("s")).collect()[0]
    check(f"3. {t}: eventos y suma bruta consistentes",
          r["e"] == FILAS_SILVER and float(r["s"]) == 241789756996.00, f"eventos={r['e']}, suma={r['s']}")

# 4) Series: frecuencia y severidad sobre calendario completo
r = g("series_frecuencia_severidad").agg(F.count("*").alias("m"), F.sum("eventos").alias("e"),
        F.round(F.sum("suma_cuantia_bruta"), 2).alias("s")).collect()[0]
check("4. series_frecuencia_severidad: 142 meses, 51.889 eventos, suma bruta consistente",
      r["m"] == 142 and r["e"] == FILAS_SILVER and float(r["s"]) == 241789756996.00,
      f"meses={r['m']}, eventos={r['e']}, suma={r['s']}")

# 5) Insumo severidad: SOLO código A con cuantía (regla VaR), valores extremos verificados
ms = g("modelo_insumo_severidad").agg(F.count("*").alias("n"), F.sum("cuantia_bruta").alias("s"),
        F.min("cuantia_bruta").alias("mn"), F.max("cuantia_bruta").alias("mx")).collect()[0]
solo_a = g("modelo_insumo_severidad").filter(F.col("codigo_perdida") != "A").count()
check("5a. modelo_insumo_severidad: 2.046 eventos, todos código A",
      ms["n"] == 2046 and solo_a == 0, f"n={ms['n']}, no-A={solo_a}")
check("5b. modelo_insumo_severidad: min=4, max=501.334.276, suma=5.335.174.360",
      ms["mn"] == 4 and ms["mx"] == 501334276 and float(ms["s"]) == 5335174360.00,
      f"min={ms['mn']}, max={ms['mx']}, suma={ms['s']}")

# 6) Insumo frecuencia: panel completo 7 x 142 = 994 filas, suma == 51.889
mf = g("modelo_insumo_frecuencia").agg(F.count("*").alias("filas"), F.sum("eventos").alias("e")).collect()[0]
check("6. modelo_insumo_frecuencia: panel 994 filas (7 categorías x 142 meses), suma eventos == 51.889",
      mf["filas"] == 994 and mf["e"] == FILAS_SILVER, f"filas={mf['filas']}, eventos={mf['e']}")

# 7) var_resultados: estructura vacía lista para la etapa de modelos
check("7. var_resultados: existe y está vacía", g("var_resultados").count() == 0)

print()
print("RESULTADO GLOBAL:", "TODAS LAS VALIDACIONES OK" if todo_ok else "HAY VALIDACIONES FALLADAS")

OK   1. freq_eventos_periodo: suma eventos == 51.889 | suma=51889
OK   1. freq_por_proceso: suma eventos == 51.889 | suma=51889
OK   1. freq_por_categoria: suma eventos == 51.889 | suma=51889
OK   1. freq_por_causa: suma eventos == 51.889 | suma=51889
OK   1. freq_por_severidad: suma eventos == 51.889 | suma=51889
OK   1b. freq_eventos_periodo: 142 meses (dic-2014 a sep-2026) | meses=142
OK   1c. freq_por_severidad: 5 niveles (BAJA/MEDIA/ALTA/EXTREMA + SIN_CUANTIA)
OK   1d. eventos código A en la serie == 2.067 | total=2067
OK   2a. ind_severidad_global: 26.701 con cuantía | 26701
OK   2b. ind_severidad_global: mediana == percentil exacto de Silver (367.371) | gold=367371.0, exacta=367371.0
OK   2c. ind_severidad_global: suma == 241.789.756.996 | 241789756996.00
OK   2d. ind_severidad_anual: suma de eventos == 26.701 | suma=26701
OK   2e. ind_severidad_anual: 13 años (2014-2026)
OK   3. perdidas_por_periodo: eventos y suma bruta consistentes | eventos=51889, suma=241789756996.0
OK   3.

In [0]:
# --- Inventario de la capa Gold y vistas de resumen ---
tablas_gold = [r["tableName"] for r in spark.sql(f"SHOW TABLES IN {SCHEMA_GOLD}").collect()]
inventario = [{"tabla": t, "filas": spark.table(f"{SCHEMA_GOLD}.{t}").count()} for t in sorted(tablas_gold)]
display(spark.createDataFrame(inventario))

print("\nIndicadores de severidad del histórico completo (26.701 eventos con cuantía):")
display(g("ind_severidad_global"))

print("\nTop 5 categorías N1 por número de eventos:")
display(g("freq_por_categoria").orderBy(F.desc("eventos")).limit(5))

filas,tabla
9,controles_snapshot
142,freq_eventos_periodo
7,freq_por_categoria
524,freq_por_causa
45,freq_por_proceso
5,freq_por_severidad
13,ind_severidad_anual
1,ind_severidad_global
994,modelo_insumo_frecuencia
2046,modelo_insumo_severidad



Indicadores de severidad del histórico completo (26.701 eventos con cuantía):


eventos_con_cuantia,suma_cuantia_bruta,promedio,mediana,p75,p90,p95,p99,desviacion_estandar,minimo,maximo,_lote_gold
26701,241789756996.00,9055456.99,367371.0,2200000.0,1.4217315E7,3.0E7,1.52E8,6.930628444E7,1.00,7000000000.00,gold_20261121_210730



Top 5 categorías N1 por número de eventos:


cod_categoria_n1,categoria_n1,eventos,eventos_con_cuantia,suma_cuantia_bruta,pct_eventos,_lote_gold
7,EJECUCIÓN Y ADMINISTRACIÓN DE PROCESOS,23218,15340,212271636895.00,44.75,gold_20261121_210730
6,FALLAS TECNOLÓGICAS,20134,7197,10113927863.00,38.8,gold_20261121_210730
2,FRAUDE EXTERNO,5599,4041,17803491880.00,10.79,gold_20261121_210730
4,"CLIENTES, PRODUCTOS Y PRÁCTICAS EMPRESARIALES",2578,10,237508924.00,4.97,gold_20261121_210730
5,DAÑOS A ACTIVOS FÍSICOS,202,29,175544340.00,0.39,gold_20261121_210730


## Registro de decisiones de la capa Gold (21-Nov-2026)

### Reglas de agregación
- **Periodicidad mensual:** `periodo` = primer día del mes (date). Calendario completo dic-2014 a sep-2026 (142 meses); meses sin eventos = 0 eventos (convención para series completas, no imputación de datos).
- **Frecuencia:** todos los eventos (51.889), con o sin cuantía; desagregada también por código A y por recuperación.
- **Severidad:** siempre sobre `cuantia_bruta`, solo eventos con cuantía (26.701), con percentiles EXACTOS (F.percentile, no aproximados). Sin cuantía ≠ severidad 0: aparece como `SIN_CUANTIA` (frecuencia) o NULL (indicadores/series).
- **Pérdidas:** sumas de cuantía bruta, neta y total recuperada; **segmento = categoría N1** (nivel BASA de mayor agregación con nombre descriptivo).
- **Insumos de modelos:** severidad SOLO `codigo_perdida='A'` con cuantía (2.046; los 21 A sin cuantía excluidos, no imputados); frecuencia como panel mensual x categoría N1 (994 filas = 7 x 142).
- **VaR:** `var_resultados` es estructura vacía (identidad automática); se llena en la etapa de modelos con método, distribuciones, nivel de confianza, horizonte, segmento, VaR y pérdida esperada.

### Tablas generadas (12)
| Tabla | Contenido | Uso principal |
| --- | --- | --- |
| freq_eventos_periodo | 142 filas | Frecuencia mensual total, con cuantía, código A y con recuperación |
| freq_por_proceso | 45 filas | Frecuencia por proceso (45 pares código-nombre) + suma de cuantía |
| freq_por_categoria | 7 filas | Frecuencia por categoría N1 |
| freq_por_causa | 524 filas | Frecuencia por código de causa |
| freq_por_severidad | 5 filas | Frecuencia por nivel de severidad derivado |
| ind_severidad_global | 1 fila | Promedio, mediana, p75-p99, desviación, extremos del histórico |
| ind_severidad_anual | 13 filas | Mismos indicadores por año |
| perdidas_por_periodo | 142 filas | Pérdidas mensuales: bruta, neta, recuperada |
| perdidas_por_segmento | 85 filas | Pérdidas anuales por categoría N1 |
| series_frecuencia_severidad | 142 filas | Serie mensual: eventos + suma, media, mediana, p95, p99 y máximo de severidad |
| modelo_insumo_severidad | 2.046 filas | Eventos A con cuantía: insumo del modelo de severidad/VaR |
| modelo_insumo_frecuencia | 994 filas | Panel mensual por categoría: insumo de distribuciones de frecuencia |
| var_resultados | 0 filas | Estructura destino de los resultados VaR |

### Verificaciones ejecutadas (todas OK)
- Suma de eventos == 51.889 en las 5 tablas de frecuencia, en pérdidas (período y segmento) y en la serie mensual.
- Suma de cuantía bruta == 241.789.756.996 COP en pérdidas por período y segmento y en la serie.
- Indicadores globales: 26.701 eventos, mediana 367.371 (percentil exacto), p99 = 152.000.000, máx 7.000.000.000 COP.
- Insumo severidad: 2.046 eventos A (todos código A), min 4, max 501.334.276, suma 5.335.174.360 COP.
- Panel de frecuencia: 994 filas (7 categorías x 142 meses), suma 51.889.
- Años incompletos documentados: 2014 (2 eventos, solo diciembre) y 2026 (hasta septiembre).